# pyKorf – Complete Feature Demonstration

**pyKorf** is a Python toolkit for programmatically reading, editing, validating, and writing
KORF hydraulic model files (`.kdf`). All modifications happen through the Python
API — no manual `.kdf` edits are required.

This notebook is aligned with:

- `README.md` (public API, persistence model, usage patterns)
- `pykorf/library/korf_manual.md` (KORF hydraulic methodology and specification rules)

## Table of Contents

1. [Section 1 – KORF Hydraulics Background](#Section-1-%E2%80%93-KORF-Hydraulics-Background)
2. [Section 2 – Package Architecture](#Section-2-%E2%80%93-Package-Architecture)
3. [Section 3 – Safety Rule](#Section-3-%E2%80%93-Safety-Rule)
4. [Setup – Imports & Paths](#Setup-%E2%80%93-Imports-&-Paths)
5. [Section 4 – Loading and Creating Models](#Section-4-%E2%80%93-Loading-and-Creating-Models)
6. [Section 5 – General Settings & Cases](#Section-5-%E2%80%93-General-Settings-&-Cases)
7. [Section 6 – Multi-Case Concepts](#Section-6-%E2%80%93-Multi-Case-Concepts)
8. [Section 7 – Working with Pipes](#Section-7-%E2%80%93-Working-with-Pipes)
9. [Section 8 – Working with Pumps & Performance Curves](#Section-8-%E2%80%93-Working-with-Pumps-&-Performance-Curves)
10. [Section 9 – Feeds & Products (Boundary Conditions)](<#Section-9-%E2%80%93-Feeds-&-Products-(Boundary-Conditions)>)
11. [Section 10 – Valves & Orifices](#Section-10-%E2%80%93-Valves-&-Orifices)
12. [Section 11 – Other Equipment Types](#Section-11-%E2%80%93-Other-Equipment-Types)
13. [Section 12 – Editing Values, Validation & Saving](#Section-12-%E2%80%93-Editing-Values,-Validation-&-Saving)
14. [Section 13 – Parser Internals](#Section-13-%E2%80%93-Parser-Internals)
15. [Section 14 – Results Extraction](#Section-14-%E2%80%93-Results-Extraction)
16. [Section 15 – KORF GUI Automation (`KorfApp`)](<#Section-15-%E2%80%93-KORF-GUI-Automation-(`KorfApp`)>)
17. [Section 16 – Full End-to-End Automation Cycle](#Section-16-%E2%80%93-Full-End-to-End-Automation-Cycle)
18. [Cleanup Section](#Cleanup-Section)


## Section 1 – KORF Hydraulics Background

Based on `pykorf/library/korf_manual.md`, KORF frames hydraulics as a specification problem:

- Pipe flow rates and many equipment inlet/outlet pressures are **unknowns**.
- KORF enforces **internal specifications** (mass balance and pressure-drop equations).
- Users provide **user specifications** (selected pressures, flows, equipment conditions).
- The key solvability rule is: **number of user specs = number of remaining unknowns**.

### Specs vs Inputs

KORF distinguishes values that count toward the solve (SPECIFIED values) from data inputs that do not
(diameter, length, properties, etc.). In a series circuit, flow should be specified on only **one** pipe.

### Multi-case simulation

Multiple scenarios can be packed into one file using semicolon-delimited values, for example `"50;55;20"`.
The marker `";C"` indicates a value calculated by KORF.

### Equipment types

| KDF Tag   | pyKorf class    | Description                                                     |
| --------- | --------------- | --------------------------------------------------------------- |
| `\PIPE`   | `Pipe`          | Process line segment with geometry, flow and hydraulic results. |
| `\PUMP`   | `Pump`          | Pump with ΔP / efficiency specs and optional performance curve. |
| `\FEED`   | `Feed`          | Inlet boundary condition entering the hydraulic network.        |
| `\PROD`   | `Product`       | Outlet boundary condition (back-pressure sink).                 |
| `\VALVE`  | `Valve`         | Control valve with ΔP, opening and Cv-based behavior.           |
| `\CHECK`  | `CheckValve`    | Non-return valve element.                                       |
| `\FO`     | `FlowOrifice`   | Restriction/orifice element with discharge behavior.            |
| `\HX`     | `HeatExchanger` | Heat exchanger pressure-drop element.                           |
| `\COMP`   | `Compressor`    | Compressor with pressure/head/power relationships.              |
| `\EXPAND` | `Expander`      | Expansion machine/turbine element.                              |
| `\JUNC`   | `Junction`      | Mixing/splitting node for branch logic and recycles.            |
| `\TEE`    | `Tee`           | Tee junction element with branch topology.                      |
| `\VESSEL` | `Vessel`        | Vessel node often used in recycle/buffer topology.              |
| `\MISC`   | `MiscEquipment` | Generic equipment pressure-drop element.                        |

Recycle circuits should include a junction/vessel path connected to a feed/product line.
KORF supports compressible fluid behavior including choked flow and can also perform process
methodology beyond hydraulics (HMB and flash calculations).


## Section 2 – Package Architecture

```text
pykorf/
├── __init__.py      ← public API: KorfModel, CaseSet, Results, open_ui
├── model.py         ← KorfModel  – top-level container
├── parser.py        ← KdfParser  – low-level CSV tokeniser/serialiser
├── cases.py         ← CaseSet    – multi-case helpers
├── results.py       ← Results    – extract calculated output values
├── automation.py    ← KorfApp    – pywinauto GUI wrapper (connect-only)
├── exceptions.py    ← KorfError, ParseError, ElementNotFound, etc.
├── utils.py         ← CSV parsing, multi-case helpers
└── elements/        ← one class per KORF element type
    ├── base.py      ← BaseElement (shared record access)
    ├── pipe.py, pump.py, feed.py, prod.py, valve.py …
```

```text
.kdf file → KdfParser.load() → list[KdfRecord] → KorfModel
                                                    ├── .general
                                                    ├── .pipes
                                                    ├── .pumps  …
                                                    (edit in memory)
                                                    → KdfParser.save() → .kdf file
```

Each element object holds a **live reference** to the parser, so updates through
`Pipe`, `Pump`, `Feed`, etc. immediately mutate the in-memory record list that will be saved.


## Section 3 – Safety Rule

**Critical:** Never launch a new KORF instance from automation code.

KORF trial/evaluation usage is limited; always reuse the already-running instance.

❌ Forbidden patterns:

- `Application().start(...)`
- `subprocess.Popen(...)` for the KORF executable
- `os.startfile(...)` on KORF

✅ Safe pattern:

- `KorfApp.connect()` (or `open_ui(...)` which internally connects)

Sections 1–14 in this notebook are file-based operations only and do not touch
the KORF executable process.


In [ ]:
# Setup – Imports & Paths
import sys, pprint
from pathlib import Path

REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pykorf import Model, KorfModel, CaseSet, Results, open_ui
from pykorf.parser import KdfParser
from pykorf.utils import split_cases, join_cases, is_calculated

SAMPLES_DIR = REPO_ROOT / "pykorf" / "library"
KDF_FILE = SAMPLES_DIR / "Pumpcases.kdf"
WORK_FILE = SAMPLES_DIR / "Pumpcases_work.kdf"
OUTPUT_FILE = SAMPLES_DIR / "Pumpcases_result.kdf"
KORF_PATH = r"C:\Program Files (x86)\Korf 36\Korf_36.exe"

print(f"pyKorf ready. Sample file: {KDF_FILE}")
print(f"Work file: {WORK_FILE}")
print(f"Output file: {OUTPUT_FILE}")
print("API aliases available:", ["Model", "KorfModel", "open_ui"])

## Section 4 – Loading and Creating Models

`Model(...)` / `KorfModel.load(...)` reads KDF data (latin-1 encoded) into memory and
builds typed collections.

Persistence behavior (from README):

- API edits change only in-memory state.
- Disk is updated only when you call `model.save()` or `model.save_as(...)`.

Creation options:

- `Model()` starts from bundled defaults (`pykorf/library/New.kdf`).
- `Model("path/to/file.kdf")` loads an existing file.

Index `0` is KORF's default template element; real process instances start at index `1`.


In [ ]:
model = KorfModel.load(KDF_FILE)
print("Model loaded:")
print(repr(model))
print("Summary:")
pprint.pprint(model.summary())

## Section 5 – General Settings & Cases

`model.general` wraps `\GEN,0,...` records.

| Property                | Meaning                           |
| ----------------------- | --------------------------------- |
| `company`               | Company metadata                  |
| `project`               | Project metadata                  |
| `units`                 | Unit system                       |
| `patm`                  | Atmospheric pressure              |
| `case_numbers`          | Active case IDs                   |
| `case_descriptions`     | Case names/descriptions           |
| `num_cases`             | Number of active cases            |
| `compressibility_model` | Gas/compressibility model setting |
| `two_phase_model`       | Two-phase model setting           |

Cases represent operating scenarios packed into one KDF.


In [ ]:
gen = model.general
general_props = {
    "company": gen.company,
    "project": gen.project,
    "units": gen.units,
    "patm": gen.patm,
    "case_numbers": gen.case_numbers,
    "case_descriptions": gen.case_descriptions,
    "num_cases": gen.num_cases,
    "compressibility_model": gen.compressibility_model,
    "two_phase_model": gen.two_phase_model,
}
print("General properties:")
pprint.pprint(general_props)

## Section 6 – Multi-Case Concepts

KORF stores multi-case values as semicolon-delimited strings.

- `split_cases('50;55;20') -> ['50', '55', '20']`
- `join_cases([50, 55, 20]) -> '50;55;20'`
- `is_calculated('100;C')` checks whether KORF marked value(s) as calculated

`CaseSet` provides case-aware helper APIs over a model.


In [ ]:
raw = "50;55;20"
parts = split_cases(raw)
joined = join_cases([50, 55, 20])
calc_demo = ";C"

print(f"split_cases({raw!r}) -> {parts}")
print(f"join_cases([50, 55, 20]) -> {joined!r}")
print(f"is_calculated({calc_demo!r}) -> {is_calculated(calc_demo)}")

In [ ]:
cases = CaseSet(model)
print("CaseSet metadata:")
print("  names   =", cases.names)
print("  numbers =", cases.numbers)
print("  count   =", cases.count)

sample_raw = model.pipe(1).flow_string
print("Sample raw flow string for pipe 1:", sample_raw)
print("Case 1 value:", cases.get_case_value(sample_raw, 1))
print("Case 2 updated string:", cases.set_case_value(sample_raw, 2, "57"))

print("Pipe flow table (first rows):")
table = cases.pipe_flows_table()
pprint.pprint(table[: min(3, len(table))])

## Section 7 – Working with Pipes

`Pipe` exposes specification and result records as properties.

### Geometry

`diameter_inch`, `schedule`, `id_m`, `length_m`, `material`, `roughness_m`, `elevation_m`, `equivalent_length_m`

### Fluid

`liquid_density`, `liquid_viscosity`

### Flow specification

`flow_string`, `get_flow()`, `flow_unit`

### Results

`velocity`, `pressure`, `pressure_drop_per_100m`, `reynolds_number`, `flow_regime`

In line with KORF methodology, specify flow on only one pipe in a series circuit.


In [ ]:
pipe1 = model.pipe(1)
pipe1_detail = {
    "name": pipe1.name,
    "description": pipe1.description,
    "diameter_inch": pipe1.diameter_inch,
    "schedule": pipe1.schedule,
    "id_m": pipe1.id_m,
    "length_m": pipe1.length_m,
    "material": pipe1.material,
    "roughness_m": pipe1.roughness_m,
    "elevation_m": pipe1.elevation_m,
    "equivalent_length_m": pipe1.equivalent_length_m,
    "liquid_density": pipe1.liquid_density,
    "liquid_viscosity": pipe1.liquid_viscosity,
    "flow_string": pipe1.flow_string,
    "get_flow": pipe1.get_flow(),
    "flow_unit": pipe1.flow_unit,
    "velocity": pipe1.velocity,
    "pressure": pipe1.pressure,
    "pressure_drop_per_100m": pipe1.pressure_drop_per_100m,
    "reynolds_number": pipe1.reynolds_number,
    "flow_regime": pipe1.flow_regime,
}
print("Pipe 1 detail:")
pprint.pprint(pipe1_detail)

In [ ]:
print("All pipe summaries:")
for idx, pipe in sorted(model.pipes.items()):
    if idx == 0:
        continue
    print(f"--- Pipe {idx} ---")
    pprint.pprint(pipe.summary())

## Section 8 – Working with Pumps & Performance Curves

Pumps can be run with either:

1. **Specified ΔP / efficiency overrides**, or
2. **Performance curve mode** (Q-H and optional efficiency/NPSH curves).

Curves typically include:

- Q-H (`curve_q`, `curve_h`)
- Q-eff (`curve_eff`)
- Q-NPSH (`curve_npsh`)

NPSH is the suction head margin needed to avoid cavitation.

Key `Pump` properties include: `pump_type`, `inlet_pipe`, `outlet_pipe`, `dp_string`, `dp_kPag`,
`efficiency_string`, `efficiency`, `power_kW`, `head_m`, `flow_m3h`, `npsh_required_m`,
`curve_q`, `curve_h`, `curve_eff`, `curve_npsh`, `has_curve`.


In [ ]:
pump1 = model.pump(1)
pump1_detail = {
    "name": pump1.name,
    "type": pump1.pump_type,
    "inlet_pipe": pump1.inlet_pipe,
    "outlet_pipe": pump1.outlet_pipe,
    "dp_string": pump1.dp_string,
    "dp_kPag": pump1.dp_kPag,
    "efficiency_string": pump1.efficiency_string,
    "efficiency": pump1.efficiency,
    "head_m": pump1.head_m,
    "flow_m3h": pump1.flow_m3h,
    "power_kW": pump1.power_kW,
    "npsh_required_m": pump1.npsh_required_m,
    "curve_q": pump1.curve_q,
    "curve_h": pump1.curve_h,
    "curve_eff": pump1.curve_eff,
    "curve_npsh": pump1.curve_npsh,
    "has_curve": pump1.has_curve,
    "summary": pump1.summary(),
}
print("Pump 1 detail:")
pprint.pprint(pump1_detail)

## Section 9 – Feeds & Products (Boundary Conditions)

Feeds are inlet boundaries and Products are outlet boundaries.
At least one pressure boundary should be specified for a solvable network.

`set_pressure()` accepts:

- list form: `[50, 55, 20]`
- semicolon string: `'50;55;20'`
- scalar: `50`


In [ ]:
print("Feed elements:")
for idx, feed in sorted(model.feeds.items()):
    if idx == 0:
        continue
    payload = {
        "name": feed.name,
        "type": feed.type,
        "elevation_m": feed.elevation_m,
        "pressure_string": feed.pressure_string,
        "pressure_kPag": feed.pressure_kPag,
        "get_pressure": feed.get_pressure(),
        "dp_string": feed.dp_string,
        "nozzle_pipe_index": feed.nozzle_pipe_index,
    }
    print(f"--- Feed {idx} ---")
    pprint.pprint(payload)
    raw_records = [r.to_line() for r in feed.records()[:3]]
    print("  Raw records sample:")
    pprint.pprint(raw_records)

print("Product elements:")
for idx, prod in sorted(model.products.items()):
    if idx == 0:
        continue
    payload = {
        "name": prod.name,
        "type": prod.type,
        "elevation_m": prod.elevation_m,
        "pressure_string": prod.pressure_string,
        "pressure_kPag": prod.pressure_kPag,
        "get_pressure": prod.get_pressure(),
        "nozzle_pipe_index": prod.nozzle_pipe_index,
    }
    print(f"--- Product {idx} ---")
    pprint.pprint(payload)
    raw_records = [r.to_line() for r in prod.records()[:3]]
    print("  Raw records sample:")
    pprint.pprint(raw_records)

## Section 10 – Valves & Orifices

Control valves may be specified by different modes depending on available data:

- direct ΔP specification
- Cv + opening behavior
- unspecified mode where KORF solves from surrounding constraints

Flow orifices are restriction elements with their own pressure-drop behavior.
KORF ΔP sign follows drawing direction (element orientation), not necessarily flow direction.


In [ ]:
if model.valves:
    print("Valves:")
    for idx, valve in sorted(model.valves.items()):
        if idx == 0:
            continue
        payload = {
            "name": valve.name,
            "valve_type": valve.valve_type,
            "cv": valve.cv,
            "dp_string": valve.dp_string,
            "dp_kPag": valve.dp_kPag,
            "opening_string": valve.opening_string,
            "inlet_pressure_kPag": valve.inlet_pressure_kPag,
            "outlet_pressure_kPag": valve.outlet_pressure_kPag,
            "connection": valve.connection,
        }
        print(f"--- Valve {idx} ---")
        pprint.pprint(payload)
else:
    print("No valves in this model.")

if model.orifices:
    print("Orifices:")
    for idx, ori in sorted(model.orifices.items()):
        if idx == 0:
            continue
        payload = {
            "name": ori.name,
            "orifice_type": ori.orifice_type,
            "dp_string": ori.dp_string,
            "dp_kPag": ori.dp_kPag,
            "beta": ori.beta,
            "bore_m": ori.bore_m,
            "discharge_coeff": ori.discharge_coeff,
            "num_holes": ori.num_holes,
            "connection": ori.connection,
        }
        print(f"--- Orifice {idx} ---")
        pprint.pprint(payload)
else:
    print("No orifices in this model.")

## Section 11 – Other Equipment Types

All typed elements inherit from `BaseElement`, which provides shared
API: `.name`, `.description`, `.notes`, and `.records()`.

This section quickly inspects available optional equipment collections.


In [ ]:
collections = [
    ("exchangers", model.exchangers),
    ("compressors", model.compressors),
    ("expanders", model.expanders),
    ("junctions", model.junctions),
    ("tees", model.tees),
    ("vessels", model.vessels),
    ("check_valves", model.check_valves),
    ("misc_equipment", model.misc_equipment),
]

for name, coll in collections:
    real_items = {idx: elem for idx, elem in coll.items() if idx > 0}
    print(f"{name}: count={len(real_items)}")
    if real_items:
        print("  names=", [elem.name for elem in real_items.values()])

## Section 12 – Editing Values, Validation & Saving

Common edit APIs:

| Area         | Methods / Properties                                                                                                      |
| ------------ | ------------------------------------------------------------------------------------------------------------------------- |
| Pipe         | `Pipe.set_flow()`, `Pipe.length_m = ...`                                                                                  |
| Feed/Product | `Feed.set_pressure()`, `Product.set_pressure()`                                                                           |
| Pump         | `Pump.set_dp()`, `Pump.set_efficiency()`, `Pump.set_curve()`                                                              |
| Valve        | `Valve.set_dp()`, `Valve.set_opening()`                                                                                   |
| CaseSet      | `CaseSet.set_pipe_flows()`, `CaseSet.set_feed_pressures()`, `CaseSet.set_product_pressures()`, `CaseSet.activate_cases()` |
| General      | `General.set_cases()`                                                                                                     |
| Any element  | `BaseElement.notes = '...'`                                                                                               |

Model-level API (README workflow):

- `model.elements` / `model.get_elements_by_type(...)`
- `model.update_element(...)`, `model.update_elements(...)`
- `model.add_element(...)`, `model.add_elements(...)`
- `model.delete_element(...)`, `model.delete_elements(...)`
- `model.copy_element(...)`, `model.copy_elements(...)`
- `model.move_element(...)`, `model.move_elements(...)`
- `model.connect_elements(...)`, `model.disconnect_elements(...)`
- `model.check_connectivity()`, `model.check_layout()`
- `model.validate()`

Save behavior:

- `model.save()` overwrites current path
- `model.save_as(path)` writes to another file

The parser preserves original line order for round-trip fidelity.


In [ ]:
work_model = KorfModel.load(KDF_FILE)
work_cases = CaseSet(work_model)

work_model.pipe(1).set_flow([60, 65, 25])
work_model.pump(1).set_efficiency(0.72)
work_cases.activate_cases([1, 2])

issues = work_model.validate()
print("Validation issues:", issues if issues else "(none)")

work_model.save_as(WORK_FILE)
verify_model = KorfModel.load(WORK_FILE)

print("Saved and reloaded:", WORK_FILE)
print("Verified pipe 1 flow:", verify_model.pipe(1).get_flow())
print("Verified pump 1 efficiency:", verify_model.pump(1).efficiency)
print("Verified active cases:", verify_model.general.case_numbers)

In [ ]:
curve_model = KorfModel.load(WORK_FILE if WORK_FILE.exists() else KDF_FILE)
curve_pump = curve_model.pump(1)
curve_pump.set_curve(
    q=[100, 150, 200, 250],
    h=[190, 170, 145, 110],
    eff=[0.62, 0.72, 0.76, 0.68],
    npsh=[2.0, 2.6, 3.5, 4.8],
)
curve_model.save_as(OUTPUT_FILE)
print("Pump curve written to:", OUTPUT_FILE)
print("Curve Q:", curve_pump.curve_q)
print("Curve H:", curve_pump.curve_h)
print("Curve eff:", curve_pump.curve_eff)
print("Curve NPSH:", curve_pump.curve_npsh)

## Section 13 – Parser Internals

`KdfRecord` fields:

- `element_type`
- `index`
- `param`
- `values`
- `raw_line`

Low-level parser access:

- `parser.get(element_type, index, param)`
- `parser.set_value(element_type, index, param, values)`
- `parser.get_all(element_type, index)`

KDF format essentials: latin-1 CSV, records like `"\TYPE",idx,"PARAM",...`,
with version header lines and `NUM` records for instance counts.


In [ ]:
parser = KdfParser(KDF_FILE)
parser.load()

print("Total records:", len(parser.records))
print("Version:", parser.version())
print("Instance counts:")
for etype in ["PIPE", "PUMP", "FEED", "PROD", "VALVE", "FO", "HX"]:
    print(f"  {etype}:", parser.num_instances(etype))

structured = [r for r in parser.records if r.element_type is not None][:10]
print("First 10 structured records:")
for rec in structured:
    print(
        {
            "element_type": rec.element_type,
            "index": rec.index,
            "param": rec.param,
            "values": rec.values,
        }
    )

In [ ]:
rec = parser.get("PIPE", 1, "TFLOW")
if rec is None:
    print("PIPE 1 TFLOW record not found.")
else:
    print("Record key:", rec.key())
    print("Record values:", rec.values)
    print("Round-trip line:", rec.to_line())

pump_records = parser.get_all("PUMP", 1)
print("First 8 PUMP,1 records:")
for r in pump_records[:8]:
    print(" ", r.to_line())

## Section 14 – Results Extraction

`Results` helper methods:

| Method                | Returns                                |
| --------------------- | -------------------------------------- |
| `pump_summary(index)` | Dict of key pump outputs               |
| `all_pump_results()`  | List of pump result dicts              |
| `pipe_velocities()`   | `{pipe_index: [velocity per case]}`    |
| `pipe_pressures()`    | `{pipe_index: [pressure per case]}`    |
| `pipe_dp_per_100m()`  | `{pipe_index: dp}`                     |
| `valve_dp()`          | `{valve_index: dp}`                    |
| `orifice_dp()`        | `{orifice_index: dp}`                  |
| `to_dataframe()`      | pandas DataFrame (if pandas installed) |

These values come from result markers in the KDF after a KORF run (often with `;C`).


In [ ]:
res_model_path = WORK_FILE if WORK_FILE.exists() else KDF_FILE
res_model = KorfModel.load(res_model_path)
res = Results(res_model)

print("Using model for results:", res_model_path)
print("pump_summary(1):")
pprint.pprint(res.pump_summary(1))
print("all_pump_results():")
pprint.pprint(res.all_pump_results())
print("pipe_velocities():")
pprint.pprint(res.pipe_velocities())
print("pipe_pressures():")
pprint.pprint(res.pipe_pressures())
print("pipe_dp_per_100m():")
pprint.pprint(res.pipe_dp_per_100m())
print("valve_dp():")
pprint.pprint(res.valve_dp())
print("orifice_dp():")
pprint.pprint(res.orifice_dp())

In [ ]:
try:
    df = res.to_dataframe()
    print("Results DataFrame preview:")
    print(df.head())
except ImportError as exc:
    print("pandas not installed; skipping DataFrame demo.")
    print(exc)

## Section 15 – KORF GUI Automation (`KorfApp`)

⚠️ These cells require KORF to already be open and available at:
`C:\Program Files (x86)\Korf 36\Korf_36.exe`.

`pywinauto` strips accelerator `&` symbols during menu matching.
Confirmed menu tree:

```text
Hy&draulics
  ├── &Title
  ├── &Cases
  ├── &Specifications
  ├── &Hydraulics
  │     ├── &Run
  │     ├── R&esume
  │     └── &Stop
  └── &Results
        ├── &View Report
        ├── &Save Report
        ├── View &RunLog
        ├── Save Run&Log
        └── View E&xcel Report
```

### `KorfApp` methods

| Method                              | Purpose                                           |
| ----------------------------------- | ------------------------------------------------- |
| `KorfApp.connect(korf_path)`        | Attach to running KORF instance (connect-only).   |
| `disconnect()`                      | Release current app handle.                       |
| `reload_model(model_path)`          | Open/reload a KDF in the running UI via Ctrl+O.   |
| `run_hydraulics()`                  | Menu action: Hydraulics → Hydraulics → Run.       |
| `wait_for_run(timeout=30)`          | Wait for Runlog popup and dismiss it.             |
| `save_model()`                      | Save current model via Ctrl+S.                    |
| `run_cycle(model_path, timeout=30)` | Convenience sequence: reload → run → wait → save. |
| `window_title`                      | Current top-window title for diagnostics.         |
| `__enter__/__exit__`                | Context-manager support.                          |

One-liner helper:

- `from pykorf import open_ui`
- `app = open_ui(WORK_FILE)`


In [ ]:
from pykorf.automation import KorfApp

korf = None
try:
    korf = KorfApp.connect(korf_path=KORF_PATH)
    print("Connected to running KORF.")
    print("Window title:", korf.window_title)
except Exception as exc:
    korf = None
    print("Could not connect to KORF. Automation cells will be skipped.")
    print(exc)

In [ ]:
if korf is not None:
    target = WORK_FILE if WORK_FILE.exists() else KDF_FILE
    korf.reload_model(target)
    print("Reloaded model in KORF:", target)
else:
    print("Skipped reload because korf is None.")

In [ ]:
if korf is not None:
    korf.run_hydraulics()
    done = korf.wait_for_run(timeout=30)
    print("Run completion status:", done)
else:
    print("Skipped run because korf is None.")

In [ ]:
if korf is not None:
    korf.save_model()
    korf.disconnect()
    print("Saved model and disconnected from KORF.")
else:
    print("Skipped save/disconnect because korf is None.")

## Section 16 – Full End-to-End Automation Cycle

Typical workflow pattern:

1. load or create model (`Model()` / `Model(path)`),
2. edit values through element/model APIs,
3. run `model.validate()` and resolve issues,
4. save work file,
5. connect to already-running KORF,
6. run hydraulics,
7. wait for run completion,
8. save in KORF,
9. disconnect,
10. reload results for extraction.

Methodology notes from the KORF manual:

- Keep user specifications balanced with unknowns.
- In a series path, specify mass flow on only one line.
- Ensure at least one pressure boundary is specified.

This pattern supports repeatable scenario execution and parameter sweeps from Python.


In [ ]:
from pykorf.automation import KorfApp


def run_scenario(
    base_file: Path, flows: list[str | float | int], out_file: Path
) -> dict:
    model = KorfModel.load(base_file)
    model.pipe(1).set_flow(flows)
    model.save_as(out_file)

    run_status = "not-run (KORF unavailable)"
    try:
        app = KorfApp.connect(korf_path=KORF_PATH)
        app.reload_model(out_file)
        app.run_hydraulics()
        done = app.wait_for_run(timeout=30)
        app.save_model()
        app.disconnect()
        run_status = f"run completed={done}"
    except Exception as exc:
        run_status = f"skipped ({exc})"

    result_model = KorfModel.load(out_file)
    result = Results(result_model).pump_summary(1)
    return {
        "file": str(out_file),
        "flows": flows,
        "status": run_status,
        "pump_summary": result,
    }


scenario_result = run_scenario(KDF_FILE, [52, 57, 24], OUTPUT_FILE)
print("Scenario result:")
pprint.pprint(scenario_result)

In [ ]:
sweep_files = []
for i, flow in enumerate([40, 50, 60, 70, 80], start=1):
    sweep_file = SAMPLES_DIR / f"sweep_{i}.kdf"
    sweep_model = KorfModel.load(KDF_FILE)
    sweep_model.pipe(1).set_flow([flow, flow + 5, max(flow - 20, 1)])
    sweep_model.save_as(sweep_file)
    sweep_files.append(str(sweep_file))

print("Parameter sweep files written (no KORF run required):")
pprint.pprint(sweep_files)

## Cleanup Section


In [ ]:
removed = []
for path in sorted(SAMPLES_DIR.glob("sweep_*.kdf")):
    path.unlink(missing_ok=True)
    removed.append(str(path))

for path in [WORK_FILE, OUTPUT_FILE]:
    if path.exists():
        path.unlink()
        removed.append(str(path))

print("Removed generated files:")
pprint.pprint(removed if removed else ["(none)"])